# Extending pyvinecopulib: custom & conditional pair copulas

The evaluators `pyvinecopulib.core.Bicop` / `Vinecop` and their PyTorch
counterparts `pyvinecopulib.torch.TorchBicop` / `TorchVinecop` are concrete
implementations of two backend-neutral contracts, `BicopLike` and `VinecopLike`.
You can plug your **own** pair copula into a vine by implementing the contract —
most easily by subclassing the canonical `BicopBase` / `VinecopBase`, which fill
in almost everything from a few primitives.

This notebook builds a small **conditional** Gaussian pair copula (its
correlation depends on covariates), hosts it in a vine, and turns the vine
**non-simplified** so each pair copula also conditions on its edge's
conditioning set. Everything here is plain NumPy — the same code runs on torch.

In [1]:
import numpy as np
from array_api_compat import array_namespace
from scipy.special import ndtr, ndtri  # standard-normal CDF / quantile

import pyvinecopulib as pv
from pyvinecopulib.core import BicopBase, VinecopBase, NonSimplifiedContext

rng = np.random.default_rng(0)

## 1. A custom pair copula

Subclass `BicopBase` and implement just three primitives — `pdf`, `hfunc1`,
`hfunc2` (the density and the two conditional CDFs). The optional second
argument `x` carries conditioning variables; here the Gaussian correlation is a
bounded, position-weighted link of them, `rho = 0.8 * tanh(scale * mean_j (j+1)
x[:, j])`, so the copula is genuinely conditional.

In [2]:
class GaussianBicop(BicopBase[np.ndarray]):
  """Gaussian pair copula whose correlation depends on covariates ``x``."""

  def __init__(self, *, scale=0.6, base_rho=0.3, rho_max=0.8):
    self._scale, self._base_rho, self._rho_max = scale, base_rho, rho_max

  def _rho(self, u, x):
    xp = array_namespace(u)
    if x is None:  # no covariates -> a fixed correlation
      return xp.full((u.shape[0],), self._base_rho, dtype=u.dtype)
    w = xp.arange(1, x.shape[1] + 1, dtype=u.dtype)  # position weights
    z = self._scale * xp.sum(x * w, axis=-1) / x.shape[1]
    return self._rho_max * xp.tanh(z)

  def pdf(self, u, x=None):
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    z1, z2 = ndtri(uc[:, 0]), ndtri(uc[:, 1])
    r = self._rho(u, x)
    om = 1 - r * r
    return np.exp(
      (2 * r * z1 * z2 - r * r * (z1**2 + z2**2)) / (2 * om)
    ) / np.sqrt(om)

  def hfunc1(self, u, x=None):  # P(U2 <= u2 | U1 = u1)
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    z1, z2 = ndtri(uc[:, 0]), ndtri(uc[:, 1])
    r = self._rho(u, x)
    return ndtr((z2 - r * z1) / np.sqrt(1 - r * r))

  def hfunc2(self, u, x=None):  # P(U1 <= u1 | U2 = u2)
    uc = np.clip(u, 1e-10, 1 - 1e-10)
    z1, z2 = ndtri(uc[:, 0]), ndtri(uc[:, 1])
    r = self._rho(u, x)
    return ndtr((z1 - r * z2) / np.sqrt(1 - r * r))

  def _draw_base_u(self, n, qrng, seeds):  # enables simulate()
    return np.random.default_rng(seeds[0] if seeds else 0).uniform(size=(n, 2))

That's it. `BicopBase` supplies the rest from those primitives: the inverse
h-functions `hinv1` / `hinv2` (numerical), a `simulate` sampler, `loglik`, a
density `plot`, and a `__repr__`.

In [3]:
cop = GaussianBicop(scale=0.8)
u = rng.uniform(0.05, 0.95, size=(5, 2))
x = rng.standard_normal(size=(5, 1))
print("rho(x):   ", np.round(cop._rho(u, x), 3))
print("pdf:      ", np.round(cop.pdf(u, x), 3))
print("loglik:   ", round(float(cop.loglik(u, x)), 3))
print("simulate: ", np.round(cop.simulate(3, seeds=[1]), 3).tolist())

rho(x):    [-0.369  0.026 -0.762 -0.139 -0.608]
pdf:       [1.122 1.055 0.084 0.988 0.714]
loglik:    -2.662
simulate:  [[0.512, 0.943], [0.144, 0.892], [0.312, 0.37]]


## 2. Hosting it in a vine

`VinecopBase` implements the whole tree-by-tree cascade (`pdf`, `rosenblatt`,
`inverse_rosenblatt`, `simulate`, `cdf`, `loglik`, `plot`, ...) on top of a few
hooks. A minimal backend just stores the pair copulas and returns them from
`_pair`:

In [4]:
class ListVinecop(VinecopBase[np.ndarray]):
  """A vine over a plain nested list of BicopLike pairs."""

  def __init__(self, pairs, structure, context):
    self._pairs = pairs
    self._bind_vine(structure, context)

  def _pair(self, tree, edge):
    return self._pairs[tree][edge]

  def _prep(self, u, name):
    return np.clip(u, 1e-10, 1 - 1e-10)

  def _draw_base_u(self, n, qrng, seeds):
    return np.random.default_rng(seeds[0] if seeds else 0).uniform(
      size=(n, self.d)
    )

## 3. A non-simplified (conditional) vine

Pass a `NonSimplifiedContext`: each pair copula `c_{a,b;D}` then receives its
edge's conditioning-set values `u_D` (assembled by the cascade) as its `x`, so
the vine is genuinely non-simplified. We build a 4-dimensional vine of
`GaussianBicop` pairs.

In [5]:
d = 4
structure = pv.RVineStructure.from_order([1, 2, 3, 4])
pairs = [
  [GaussianBicop(scale=0.6) for _ in range(d - 1 - t)] for t in range(d - 1)
]
vine = ListVinecop(pairs, structure, NonSimplifiedContext())
print(vine)
print("dimension:", vine.dim, " trees:", vine.trunc_lvl)

ListVinecop(dim=4, trunc_lvl=3, order=[1, 2, 3, 4])
dimension: 4  trees: 3


Because the conditioning variables are finalised before they are needed, the
non-simplified transform is an exact bijection — `inverse_rosenblatt` inverts
`rosenblatt`:

In [6]:
U = rng.uniform(0.05, 0.95, size=(500, d))
W = vine.rosenblatt(U)  # dependent -> independent uniforms
U_back = vine.inverse_rosenblatt(W)  # and back
print("round-trip max error:", float(np.abs(U_back - U).max()))
print("log-likelihood:      ", round(float(vine.loglik(U)), 2))

round-trip max error: 1.817990202823694e-15
log-likelihood:       -41.85


External covariates work too: pass `x` (row-aligned with `u`) to any evaluator
and every pair sees it appended to its conditioning matrix. The density shifts
with the covariates:

In [7]:
X = rng.standard_normal(size=(500, 2))
print(
  "mean pdf, x = 0 :", round(float(vine.pdf(U, x=np.zeros((500, 2))).mean()), 3)
)
print("mean pdf, x ~ N :", round(float(vine.pdf(U, x=X).mean()), 3))

mean pdf, x = 0 : 1.011
mean pdf, x ~ N : 1.045


## What's next

- The same subclassing pattern works on PyTorch: implement the primitives with
  `torch` ops (and make the class an `nn.Module`) to get autograd and GPU.
- See `pyvinecopulib.core.BicopBase` / `VinecopBase`, `BicopLike` /
  `VinecopLike`, and `NonSimplifiedContext` for the full contracts, and
  `VinecopBase.sequential_fit` to *fit* a non-simplified vine edge by edge.